# AIGC 5005 · Week 2 · Part 5
## Files, JSON and web APIs

Companion notebook to Part 5 of the Week 2 learning module. Every example from that part is here, ready to run, with comments explaining what to notice.

**Estimated time:** 60 minutes

**How to use it:** read each explanation, run the cell beneath it, and compare the output with what you expected *before* you ran it. The prediction is where the learning happens. Change values and run cells again; you cannot break anything.

**Kernel:** select the kernel from your `week02-records` virtual environment (top right in VS Code or Jupyter). If VS Code offers to install `ipykernel`, accept.

## 0. Check that `requests` is installed

This part needs the `requests` package in **the environment your kernel is using**. If the cell below reports it missing, uncomment the `%pip` line and run it. `%pip` (with the percent sign) installs into the kernel's own environment, which plain `!pip` does not guarantee.

In [1]:
# %pip install requests

try:
    import requests
    print("requests", requests.__version__, "is available.")
except ModuleNotFoundError:
    print("requests is NOT installed in this kernel's environment.")
    print("Uncomment the %pip line above, run this cell, then restart the kernel.")

requests 2.32.5 is available.


## 1. Reading and writing text files

Always use a `with` block: it closes the file even if an error occurs partway. Always pass `encoding="utf-8"`; the default differs between Windows and macOS.

These cells create files in the same folder as this notebook.

In [2]:
# "w" creates the file, or ERASES it if it already exists.
with open("notes.txt", "w", encoding="utf-8") as f:
    f.write("first line\n")
    f.write("second line\n")

# "a" appends to the end instead.
with open("notes.txt", "a", encoding="utf-8") as f:
    f.write("appended line\n")

# The default mode "r" reads. Iterating a file gives one line at a time.
with open("notes.txt", encoding="utf-8") as f:
    for line in f:
        print(repr(line), "->", line.strip())   # repr shows the hidden \n that strip removes

'first line\n' -> first line
'second line\n' -> second line
'appended line\n' -> appended line


## 2. Paths with `pathlib`

Joining strings with `/` breaks on Windows. `pathlib` handles it, and offers one-line read and write for whole files.

In [3]:
from pathlib import Path

data_dir = Path("data")
output = data_dir / "summary.txt"          # / joins paths correctly on every platform

data_dir.mkdir(exist_ok=True)              # exist_ok avoids an error on re-runs
output.write_text("done\n", encoding="utf-8")

print("path   :", output)
print("exists :", output.exists())
print("content:", output.read_text(encoding="utf-8").strip())

path   : data\summary.txt
exists : True
content: done


## 3. JSON

JSON maps almost directly onto Python: objects become dicts, arrays become lists, `true`/`false`/`null` become `True`/`False`/`None`.

Functions ending in **s** (`dumps`, `loads`) work with **s**trings. Those without (`dump`, `load`) work with files.

In [4]:
import json

record = {"name": "Ada", "scores": [91, 88], "active": True, "mentor": None}

text = json.dumps(record, indent=2)         # Python to JSON text
print(text)                                 # note: True -> true, None -> null

back = json.loads(text)                     # JSON text back to Python
print("\nround trip equal:", back == record)

{
  "name": "Ada",
  "scores": [
    91,
    88
  ],
  "active": true,
  "mentor": null
}

round trip equal: True


In [5]:
# Straight to and from a file.
with open("record.json", "w", encoding="utf-8") as f:
    json.dump(record, f, indent=2)

with open("record.json", encoding="utf-8") as f:
    loaded = json.load(f)

print(loaded)

{'name': 'Ada', 'scores': [91, 88], 'active': True, 'mentor': None}


### Watch out: JSON keys are always strings

Integer keys do not survive a round trip. This bites in the worked example below.

In [6]:
original = {1: "a", 2: "b"}
reloaded = json.loads(json.dumps(original))

print("original keys:", list(original), "->", type(list(original)[0]).__name__)
print("reloaded keys:", list(reloaded), "->", type(list(reloaded)[0]).__name__)

print("reloaded.get(1)  :", reloaded.get(1))      # None: the integer 1 is not a key any more
print("reloaded.get('1'):", reloaded.get("1"))    # 'a'

original keys: [1, 2] -> int
reloaded keys: ['1', '2'] -> str
reloaded.get(1)  : None
reloaded.get('1'): a


## 4. Calling a web API

Open https://jsonplaceholder.typicode.com/todos in your browser first, to see the shape of the data.

Two rules: **always set a timeout** (otherwise a silent server hangs your program forever), and **always call `raise_for_status()`** (a failed request still returns a response object).

In [7]:
URL = "https://jsonplaceholder.typicode.com/todos"

try:
    response = requests.get(URL, timeout=10)
    response.raise_for_status()             # raises HTTPError on 404, 500 and so on
    todos = response.json()                 # parsed straight into lists and dicts
    print("status :", response.status_code)
    print("records:", len(todos))
    print("first  :", todos[0])
except requests.RequestException as error:
    # RequestException covers timeouts, connection failures and HTTP errors.
    print("Request failed:", error)
    print("Check your internet connection. The worked example below still runs offline.")

status : 200
records: 200
first  : {'userId': 1, 'id': 1, 'title': 'delectus aut autem', 'completed': False}


## 5. Worked example

The same program as in the module, split into cells so you can run and inspect each function.

**One notebook-only addition:** if the download fails, `main()` falls back to a small built-in sample, so that Run All still works without internet. The module's script version does not have this, and it would be a reasonable improvement to make to yours.

In [8]:
import json
from pathlib import Path

import requests

URL = "https://jsonplaceholder.typicode.com/todos"
OUTPUT = Path("summary.json")

# Offline fallback: a few records with the same shape as the real API.
SAMPLE_TODOS = [
    {"userId": 1, "id": 1, "title": "a", "completed": True},
    {"userId": 1, "id": 2, "title": "b", "completed": False},
    {"userId": 2, "id": 3, "title": "c", "completed": True},
    {"userId": 2, "id": 4, "title": "d", "completed": True},
    {"userId": 3, "id": 5, "title": "e", "completed": False},
]

In [9]:
def fetch_todos(url):
    """Download the to-do records and return them as Python objects."""
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()

In [10]:
def completion_by_user(todos):
    """Return a dict mapping each user id to a (completed, total) tuple."""
    counts = {}
    for todo in todos:
        user = todo["userId"]
        # The counting pattern, with a tuple default instead of 0.
        done, total = counts.get(user, (0, 0))
        # int(True) is 1 and int(False) is 0, so this adds 1 only if completed.
        counts[user] = (done + int(todo["completed"]), total + 1)
    return counts


# Test it on the sample before trusting it with real data.
print(completion_by_user(SAMPLE_TODOS))     # {1: (1, 2), 2: (2, 2), 3: (0, 1)}

{1: (1, 2), 2: (2, 2), 3: (0, 1)}


In [11]:
def completion_rates(counts):
    """Convert (completed, total) pairs into percentages."""
    # A dict comprehension that unpacks each (done, total) tuple as it goes.
    return {user: round(100 * done / total, 1)
            for user, (done, total) in counts.items()}


print(completion_rates(completion_by_user(SAMPLE_TODOS)))   # {1: 50.0, 2: 100.0, 3: 0.0}

{1: 50.0, 2: 100.0, 3: 0.0}


In [12]:
def main():
    try:
        todos = fetch_todos(URL)
    except requests.RequestException as error:
        print(f"Download failed ({error}); using built-in sample data.")
        todos = SAMPLE_TODOS

    counts = completion_by_user(todos)
    rates = completion_rates(counts)

    # max with key=rates.get returns the KEY whose VALUE is largest.
    # Careful: if several users tie for the top rate, max silently returns
    # whichever comes first. With the real data, users 5 and 10 are tied.
    best = max(rates, key=rates.get)
    summary = {
        "records": len(todos),
        "users": len(rates),
        "rates": rates,
        "most_complete_user": best,
    }

    OUTPUT.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(f"Read {len(todos)} records for {len(rates)} users.")
    print(f"User {best} has the highest completion rate at {rates[best]}%.")
    print(f"Summary written to {OUTPUT}")


# In a .py script, this guard stops main() running when the file is imported.
# In a notebook __name__ is always "__main__", so it simply runs.
if __name__ == "__main__":
    main()

Read 200 records for 10 users.
User 5 has the highest completion rate at 60.0%.
Summary written to summary.json


### Inspect the output file

Look at the keys under `"rates"`. They were integers in Python; in the file they are strings.

In [13]:
print(OUTPUT.read_text(encoding="utf-8"))

reloaded = json.loads(OUTPUT.read_text(encoding="utf-8"))
first_key = next(iter(reloaded["rates"]))
print(f"First rate key after reloading: {first_key!r} ({type(first_key).__name__})")

{
  "records": 200,
  "users": 10,
  "rates": {
    "1": 55.0,
    "2": 40.0,
    "3": 35.0,
    "4": 30.0,
    "5": 60.0,
    "6": 30.0,
    "7": 45.0,
    "8": 55.0,
    "9": 40.0,
    "10": 60.0
  },
  "most_complete_user": 5
}
First rate key after reloading: '1' (str)


### Look again: is user 5 really the best?

Scan the rates above. The program reported one winner, but the data may disagree. The cell below finds **every** user at the top rate, which is what an honest summary should report.

In [14]:
# Read the rates back from the file so this cell does not depend on main()'s local variables.
file_rates = json.loads(OUTPUT.read_text(encoding="utf-8"))["rates"]
top_rate = max(file_rates.values())
leaders = [user for user, rate in file_rates.items() if rate == top_rate]

print(f"Top rate: {top_rate}%")
print(f"Users at that rate: {leaders}")
if len(leaders) > 1:
    print("A tie. max() reported only the first of these.")

Top rate: 60.0%
Users at that rate: ['5', '10']
A tie. max() reported only the first of these.


---
## Try it: Extend the summary

1. Add a field to the summary listing the users whose completion rate is **below 50%**, using a list comprehension.
2. Run it, check `summary.json`, then make the same change in your `summarize_todos.py` script, commit and push.

Start here:

In [15]:
todos = SAMPLE_TODOS
rates = completion_rates(completion_by_user(todos))

# below_half = [ ... ]
# print(below_half)      # with the sample data, expected: [3]

## Clean up (optional)

Removes the files this notebook created, if you want a tidy folder.

In [16]:
# import shutil

# for name in ["notes.txt", "record.json", "summary.json"]:
#     Path(name).unlink(missing_ok=True)
# shutil.rmtree("data", ignore_errors=True)
# print("Cleaned up.")